In [1]:
from app import get_python_assistant
from langchain.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
from dotenv import load_dotenv
import os
from fastapi import HTTPException
from pydantic import BaseModel
from pprint import pprint

#### Documents

In [2]:
Documents = os.listdir("files")
Documents

['fastapi_tutorial.pdf',
 'Interview-level-QA-on-Python-Programming.pdf',
 'Introduction_to_Python_Programming.pdf',
 'learn-web-development-python-hands.pdf',
 'python_basics.pdf']

In [3]:
for d in Documents:
    print("documnets used:\n")
    print("-", d)

documnets used:

- fastapi_tutorial.pdf
documnets used:

- Interview-level-QA-on-Python-Programming.pdf
documnets used:

- Introduction_to_Python_Programming.pdf
documnets used:

- learn-web-development-python-hands.pdf
documnets used:

- python_basics.pdf


#### Initializing the assistant

In [4]:
assistant = get_python_assistant()

if not hasattr(assistant, 'builder'):
    raise ValueError("assistant.builder is not defined")

print("Assistant initialized")
print("Vector store documents:", assistant.vector_store._collection.count())

[VectorStore] Loaded 2850 chunks
Assistant initialized
Vector store documents: 2850


#### Test Retrieval Tool

In [5]:
tool = assistant.tools[0]

query = "How does FastAPI dependency injection work?"
context = tool.invoke(query)

print("Retrieved context:\n")
print(context[:1500])


Retrieved context:

Source: fastapi_tutorial.pdf | Page: 80

22. FastAPI – DeFpasetAPnI d– Peytnhocni Weesb Framework
The built-in dependency injection system of FastAPI makes it possible to
integrate components easier when building your API. In programming,
Dependency injection refers to the mechanism where an object receives
other objects that it depends on. The other objects are called dependencies.
Dependency injection has the following advantages:
 reuse the same shared logic
 share database connections
 enforce authentication and security features
Assuming that a FastAPI app has two operation functions both having the
same query parameters id, name and age.
from fastapi import FastAPI
app = FastAPI()
@app.get("/user/")
async def user(id: str, name: str, age: int):
return {"id": id, "name": name, "age": age}
@app.get("/admin/")
async def admin(id: str, name: str, age: int):
return {"id": id, "name": name, "age": age}
In case of any changes such as adding/removing query paramete

In [6]:
load_dotenv()





def ask_python_question(
    question: str,
):
    """Ask a coding related question"""
    try:
        
        assistant = get_python_assistant()
        # user_id = f"user_{user_info['user_id']}"
        user_id = f"user_1"

        
        response = assistant.ask_question(
            question=question,
            user_id=user_id
        )
        
        return {
            "success": True,
            f"\nuser_id": user_id,
            f"\nquestion": question,
            f"\nanswer": response
        }
        
    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"Error processing question: {str(e)}"
        )


In [7]:
def ask_with_trace(question: str, user_id: str = "notebook"):
    """
    Ask a question and return:
    - final answer
    - whether retrieval was used
    - tool inputs / outputs (trace)
    """

    agent = assistant.builder.compile(
        checkpointer=assistant.checkpointer
    )

    used_retrieval = False
    tool_calls = []
    final_answer = ""

    for event in agent.stream(
        {"messages": [HumanMessage(content=question)]},
        {"configurable": {"thread_id": user_id}},
    ):
        for node, value in event.items():
            if not isinstance(value, dict):
                continue

            messages = value.get("messages", [])
            if not messages:
                continue

            msg = messages[-1]

            # Tool call detection
            if isinstance(msg, AIMessage) and msg.tool_calls:
                used_retrieval = True
                tool_calls.extend(msg.tool_calls)

            # Tool output
            if isinstance(msg, ToolMessage):
                tool_calls.append({
                    "tool": msg.name,
                    "output_preview": msg.content[:300]
                })

            # Final answer
            if isinstance(msg, AIMessage) and msg.content:
                final_answer = msg.content

    return {
        "question": question,
        "answer": final_answer,
        "used_retrieval": used_retrieval,
        "tool_trace": tool_calls,
    }


#### Testing Queries

***No Retrieval Needed***

In [8]:
result = ask_with_trace("What is a Python?")
pprint(result)


{'answer': 'Python is a high-level, interpreted programming language known for '
           'its readability and simplicity. It supports multiple programming '
           'paradigms, including procedural, object-oriented, and functional '
           'programming. Python is widely used for web development, data '
           'analysis, artificial intelligence, scientific computing, and more.',
 'question': 'What is a Python?',
 'tool_trace': [],
 'used_retrieval': False}


In [9]:
result = ask_with_trace("What is a Python list?")
pprint(result)


{'answer': 'A Python list is a built-in data structure that allows you to '
           'store an ordered collection of items. Lists can hold a variety of '
           'data types, including numbers, strings, and other objects. They '
           'are mutable, meaning you can change their content after creation, '
           'and they are defined using square brackets, e.g., `my_list = [1, '
           "2, 3, 'apple']`.",
 'question': 'What is a Python list?',
 'tool_trace': [],
 'used_retrieval': False}


In [10]:
result = ask_with_trace("can i use python for backend?")
pprint(result)


{'answer': 'Yes, you can use Python for backend development. Python is a '
           'versatile language that supports various web frameworks, such as '
           'Django and Flask, which are commonly used to build server-side '
           'applications. These frameworks provide tools and libraries to '
           'handle web requests, manage databases, and create RESTful APIs, '
           'making Python a popular choice for backend development.',
 'question': 'can i use python for backend?',
 'tool_trace': [{'args': {'query': 'using python for backend development'},
                 'id': 'call_3YW8VhMAs9gu2Vd08ifZylah',
                 'name': 'retriever_tool',
                 'type': 'tool_call'},
                {'output_preview': 'Source: '
                                   'learn-web-development-python-hands.pdf | '
                                   'Page: 3\n'
                                   '\n'
                                   'Learn Web Development with Python\n'


In [11]:
result = ask_with_trace("is python a fullstack language?")
pprint(result)


{'answer': 'Python is not inherently a "fullstack" language, but it can be '
           'used for fullstack development. Fullstack development refers to '
           'the ability to work on both the frontend (client-side) and backend '
           '(server-side) of a web application. While Python is primarily used '
           'for backend development (with frameworks like Django and Flask), '
           'it can be combined with frontend technologies (like HTML, CSS, and '
           'JavaScript) to create complete web applications. Therefore, while '
           'Python itself is not a fullstack language, it can be part of a '
           'fullstack development process.',
 'question': 'is python a fullstack language?',
 'tool_trace': [{'args': {'query': 'is python a fullstack language'},
                 'id': 'call_A8kh44n8k65h2e0p0Z5LvgfZ',
                 'name': 'retriever_tool',
                 'type': 'tool_call'},
                {'output_preview': 'Source: '
                   

In [12]:
result = ask_with_trace("What is a javasscript?")
pprint(result)


{'answer': "I'm sorry, but I am only a Python learning assistant and can only "
           'provide information related to Python. If you have any questions '
           'about Python, feel free to ask!',
 'question': 'What is a javasscript?',
 'tool_trace': [],
 'used_retrieval': False}


***Needs Retrieval***


In [13]:
result = ask_with_trace("What does PEP 8 with recommend for maximum line length? include citations from database")
pprint(result)


{'answer': 'PEP 8 recommends a maximum line length of 79 characters. This '
           'guideline is intended to enhance the readability of code. It is '
           'common practice to adhere to this limit to ensure that code can be '
           'easily viewed in various environments, such as terminals and text '
           'editors.',
 'question': 'What does PEP 8 with recommend for maximum line length? include '
             'citations from database',
 'tool_trace': [{'args': {'query': 'PEP 8 maximum line length recommendation'},
                 'id': 'call_Vy93Wk6rLMST0gWdngUia694',
                 'name': 'retriever_tool',
                 'type': 'tool_call'},
                {'output_preview': 'Source: '
                                   'learn-web-development-python-hands.pdf | '
                                   'Page: 66\n'
                                   '\n'
                                   "PEP 8, we can avoid this. I'm such a fan "
                                

In [17]:
result = ask_with_trace("Explain django for backend development")
pprint(result)


{'answer': 'Django is a high-level Python web framework that encourages rapid '
           'development and clean, pragmatic design. It is particularly '
           'well-suited for backend development due to its robust features and '
           'built-in functionalities. Here are some key aspects of using '
           'Django for backend development:\n'
           '\n'
           '1. **MVC Architecture**: Django follows the Model-View-Controller '
           '(MVC) architectural pattern, which helps in organizing code and '
           'separating concerns. In Django, this is often referred to as '
           'Model-View-Template (MVT).\n'
           '\n'
           '2. **ORM (Object-Relational Mapping)**: Django includes a powerful '
           'ORM that allows developers to interact with the database using '
           'Python code instead of SQL. This makes database operations more '
           'intuitive and reduces the likelihood of SQL injection attacks.\n'
           '\n'
      

In [15]:
result = ask_with_trace("How do I use the new API endpoint for user authentication?")
pprint(result)

{'answer': 'To create a new API endpoint for user authentication in FastAPI, '
           'you can follow these general steps:\n'
           '\n'
           '1. **Install FastAPI and Required Libraries**: Make sure you have '
           'FastAPI and an ASGI server like `uvicorn` installed. You can '
           'install them using pip:\n'
           '\n'
           '   ```bash\n'
           '   pip install fastapi uvicorn python-multipart\n'
           '   ```\n'
           '\n'
           '2. **Create the FastAPI Application**: Set up your FastAPI '
           'application and define the authentication endpoint.\n'
           '\n'
           "3. **Define the Authentication Logic**: You can use FastAPI's "
           'dependency injection system to handle authentication.\n'
           '\n'
           'Here’s a simple example of how to create a user authentication '
           'endpoint:\n'
           '\n'
           '```python\n'
           'from fastapi import FastAPI, Depends, HTTPExc

In [19]:
result = ask_with_trace("teach me about alembic")
pprint(result)

{'answer': 'Alembic is a lightweight database migration tool for use with '
           'SQLAlchemy, the popular Python Object Relational Mapper (ORM). It '
           'is designed to handle database schema changes in a systematic and '
           'version-controlled manner. Here are some key features and concepts '
           'related to Alembic:\n'
           '\n'
           '1. **Database Migrations**: Alembic allows you to create and '
           'manage migrations, which are scripts that define changes to the '
           'database schema. This includes creating, modifying, or deleting '
           'tables and columns.\n'
           '\n'
           '2. **Version Control**: Each migration is assigned a unique '
           'version identifier, allowing you to track changes over time. You '
           'can apply or revert migrations as needed, making it easier to '
           'manage database schema changes in development and production '
           'environments.\n'
           '\n'
 

In [20]:
result = ask_with_trace("give me the citations for alembic")
pprint(result)

{'answer': 'Here are the citations related to Alembic from the provided '
           'sources:\n'
           '\n'
           '1. **Learn Web Development with Python**: This source discusses '
           'various aspects of database management and migrations, including '
           'the use of SQLAlchemy and Alembic for handling database schema '
           'changes. (Source: learn-web-development-python-hands.pdf | Page: '
           '772)\n'
           '\n'
           '2. **Python Basics**: This document may contain references to '
           'database operations and the use of Alembic in conjunction with '
           'SQLAlchemy for managing migrations. (Source: python_basics.pdf | '
           'Page: 34)\n'
           '\n'
           'These citations provide context for understanding how Alembic is '
           'utilized in Python applications, particularly in conjunction with '
           'SQLAlchemy for database migrations.',
 'question': 'give me the citations for alembic',
 'too